In [82]:
import pandas as pd 
import numpy as np
import pyarrow
cafe_data=pd.read_csv('dirty_cafe_sales.csv')
#* first we rename the columns
cafe_data.rename(columns={})
cafe_data.columns=cafe_data.columns.str.strip().str.lower().str.replace(' ','_')
cafe_data



,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2,2.0,4.0,NaN,UNKNOWN,2023-08-30
9996,TXN_9659401,NaN,3,NaN,3.0,Digital Wallet,NaN,2023-06-02
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3,NaN,3.0,Digital Wallet,NaN,2023-12-02


In [83]:
 #* Now we handle the garbage values('ERROR','UNKNOWN')
cafe_data=cafe_data.replace(['UNKNOWN','ERROR'],np.nan)
cafe_data

,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,NaN,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,NaN,NaN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2,2.0,4.0,NaN,NaN,2023-08-30
9996,TXN_9659401,NaN,3,NaN,3.0,Digital Wallet,NaN,2023-06-02
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3,NaN,3.0,Digital Wallet,NaN,2023-12-02


In [84]:
 #* Now we analyze and change data types 
# print(cafe_data.dtypes)
cafe_data['quantity']=pd.to_numeric(cafe_data['quantity'],errors='coerce')
cafe_data['price_per_unit']=pd.to_numeric(cafe_data['price_per_unit'],errors='coerce')
cafe_data['total_spent']=pd.to_numeric(cafe_data['total_spent'],errors='coerce')
#* for datetime columns
cafe_data['transaction_date']=pd.to_datetime(cafe_data['transaction_date'],errors='coerce')
print(cafe_data.dtypes)

transaction_id                 str
item                           str
quantity                   float64
price_per_unit             float64
total_spent                float64
payment_method                 str
location                       str
transaction_date    datetime64[us]
dtype: object


In [ ]:
# cafe_data['item'].value_counts()
#? Check if each item has a consistent price 
#todo we fill price_per_unit based on items known price
cafe_data['item']=cafe_data['item'].fillna('UNKNOWN')

#TODO Fill price_per_unit in two ways:
#TODO 1. Known items → use mode of that item's price (fixed price per item)
#TODO 2. Unknown items → calculate from total_spent / quantity
#* 1 — Fill known items using mode
cafe_data.loc[cafe_data['item'] != 'UNKNOWN', 'price_per_unit'] = cafe_data[cafe_data['item'] != 'UNKNOWN'].groupby('item')['price_per_unit'].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else np.nan)
)
#* 2 — Fill Unknown items using total_spent / quantity
cafe_data.loc[cafe_data['item'] == 'UNKNOWN', 'price_per_unit'] = cafe_data.loc[cafe_data['item'] == 'UNKNOWN', 'price_per_unit'].fillna(
    cafe_data['total_spent'] / cafe_data['quantity']
)
#TODO Fill total_spent first using quantity × price_per_unit
cafe_data['total_spent'] = cafe_data['total_spent'].fillna(
    cafe_data['quantity'] * cafe_data['price_per_unit']
)

# TODO TO HANDLE NULL VALUES IN QUANTITY WE DIVIDE TOTAL_SPENT WITH PRICE PER UNIT and see if stil any null value exists 
cafe_data['quantity']=cafe_data['quantity'].fillna(cafe_data['total_spent']/cafe_data['price_per_unit'])
 
 #*NOW WE DROP ROWS WHERE TOTAL_SPEND AND QUANTITY ARE SIMULTANEOUSLY empty

cafe_data=cafe_data.dropna(subset=['quantity', 'total_spent'], how='all')

print("Total spent nulls remaining:", cafe_data['total_spent'].isnull().sum())
print("quantity nulls remaining:", cafe_data['quantity'].isnull().sum())
print(cafe_data.groupby('item')['price_per_unit'].unique())
print("Price per unit nulls remaining:", cafe_data['price_per_unit'].isnull().sum())#* successfully handles null values in price_per_unit


Total spent nulls remaining: 3
quantity nulls remaining: 3
item
Cake                                      [3.0]
Coffee                                    [2.0]
Cookie                                    [1.0]
Juice                                     [3.0]
Salad                                     [5.0]
Sandwich                                  [4.0]
Smoothie                                  [4.0]
Tea                                       [1.5]
UNKNOWN     [3.0, 1.5, 2.0, 1.0, 5.0, 4.0, nan]
Name: price_per_unit, dtype: object
Price per unit nulls remaining: 6


In [ ]:
 #?DEALING WITH NULL VALUES IN LOCATION ,TRANSACTION_DATE AND PAYMENT_METHOD
# print(cafe_data['payment_method'].isnull().sum()/cafe_data['payment_method'].size) #*31% values are null
# print(cafe_data['location'].isnull().sum()/cafe_data['location'].size) #*39% values are null
# print(cafe_data['transaction_date'].isnull().sum()/cafe_data['transaction_date'].size) #*4% values are null

# mask = (cafe_data['payment_method'].isnull() &  cafe_data['location'].isnull() & cafe_data['transaction_date'].isnull())

# print("Rows where all three are null:", mask.sum())
# print("Total spent in these rows",cafe_data[mask]['total_spent'].sum())
# print("\nOverall total spent:")
# print(cafe_data['total_spent'].sum())
# print("\nPercentage of revenue these rows represent:")
# print(cafe_data[mask]['total_spent'].sum() / cafe_data['total_spent'].sum() * 100, "%")
#* safe to drop these where all 3 are empty simultaneously

cafe_data=cafe_data.dropna(subset=['payment_method','location','transaction_date'],how='all')
#!WE STILL HAVE 6 ROWS WHERE ITEM IS UNKNOWN,PRICE_PER_UNIT IS MISSING AND IN THESE 6 EITHER QUANTITY IS MISSING OR TOTAL_SPENT IS MISSING
#! SINCE NO LOGIC WORK SO WE DROP THESE
cafe_data = cafe_data.dropna(subset=['price_per_unit'])


#* to handle null values in date we ffill()
cafe_data = cafe_data.sort_values('transaction_date').reset_index(drop=True)
cafe_data['transaction_date']=cafe_data['transaction_date'].ffill()
print( cafe_data['transaction_date'].isnull().sum())# handled


#TODO Fill payment_method nulls with Unknown (31% null)
cafe_data['payment_method'] = cafe_data['payment_method'].fillna('UNKNOWN')
print(cafe_data['payment_method'].isnull().sum())


#TODO Fill location nulls with Unknown (39% null, too many to fabricate)
cafe_data['location'] = cafe_data['location'].fillna('UNKNOWN')
print(cafe_data['location'].isnull().sum())

0
0
0


In [87]:
# cafe_data.head(50 )
print(cafe_data.isnull().sum())
cafe_data['month'] = cafe_data['transaction_date'].dt.strftime('%B')
cafe_data['day_of_week'] = cafe_data['transaction_date'].dt.day_name()
cafe_data['is_weekend'] = cafe_data['transaction_date'].dt.dayofweek >= 5
cafe_data

transaction_id      0
item                0
quantity            0
price_per_unit      0
total_spent         0
payment_method      0
location            0
transaction_date    0
dtype: int64


,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date,month,day_of_week,is_weekend
0,TXN_5358805,Coffee,5.0,2.0,10.0,Digital Wallet,UNKNOWN,2023-01-01,January,Sunday,True
1,TXN_1604072,Coffee,2.0,2.0,4.0,UNKNOWN,UNKNOWN,2023-01-01,January,Sunday,True
2,TXN_8249251,Cake,3.0,3.0,9.0,UNKNOWN,In-store,2023-01-01,January,Sunday,True
3,TXN_2024598,Sandwich,1.0,4.0,4.0,Digital Wallet,In-store,2023-01-01,January,Sunday,True
4,TXN_3093284,Coffee,4.0,2.0,8.0,Digital Wallet,In-store,2023-01-01,January,Sunday,True
...,...,...,...,...,...,...,...,...,...,...,...
9904,TXN_4659954,UNKNOWN,3.0,4.0,12.0,Credit Card,In-store,2023-12-31,December,Sunday,True
9905,TXN_8104914,Cookie,1.0,1.0,1.0,UNKNOWN,Takeaway,2023-12-31,December,Sunday,True
9906,TXN_9460419,Cake,1.0,3.0,3.0,UNKNOWN,Takeaway,2023-12-31,December,Sunday,True
9907,TXN_3130865,Juice,3.0,3.0,9.0,UNKNOWN,In-store,2023-12-31,December,Sunday,True


In [88]:
print("="*50)
print("FINAL VALIDATION REPORT")
print("="*50)

print(f"\n✅ Shape: {cafe_data.shape}")
print(f"✅ Rows remaining: {len(cafe_data)} / 10000")
print(f"✅ Rows removed: {10000 - len(cafe_data)}")

print("\n--- Null Counts ---")
print(cafe_data.isnull().sum())

print("\n--- Data Types ---")
print(cafe_data.dtypes)

print("\n--- Duplicate Rows ---")
print("Duplicates:", cafe_data.duplicated().sum())

print("\n--- Numeric Columns Stats ---")
print(cafe_data[['quantity', 'price_per_unit', 'total_spent']].describe())

print("\n--- Unique Values in Categorical Columns ---")
print("Items:", cafe_data['item'].unique())
print("Payment Methods:", cafe_data['payment_method'].unique())
print("Locations:", cafe_data['location'].unique())

print("\n--- Date Range ---")
print("From:", cafe_data['transaction_date'].min())
print("To:  ", cafe_data['transaction_date'].max())

print("\n" + "="*50)
print("CLEANING COMPLETE ✅")
print("="*50)

FINAL VALIDATION REPORT

✅ Shape: (9909, 11)
✅ Rows remaining: 9909 / 10000
✅ Rows removed: 91

--- Null Counts ---
transaction_id      0
item                0
quantity            0
price_per_unit      0
total_spent         0
payment_method      0
location            0
transaction_date    0
month               0
day_of_week         0
is_weekend          0
dtype: int64

--- Data Types ---
transaction_id                 str
item                           str
quantity                   float64
price_per_unit             float64
total_spent                float64
payment_method                 str
location                       str
transaction_date    datetime64[us]
month                          str
day_of_week                    str
is_weekend                    bool
dtype: object

--- Duplicate Rows ---
Duplicates: 0

--- Numeric Columns Stats ---
          quantity  price_per_unit  total_spent
count  9909.000000     9909.000000  9909.000000
mean      3.025230        2.946110     8.9255

In [89]:
cafe_data.to_csv('cafe_sales_cleaned.csv', index=False)